In [1]:
import pandas as pd, glob
f = glob.glob("../data/raw/sparcs/2022/*.csv")[0]
df = pd.read_csv(f, dtype=str, low_memory=False)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df.to_parquet("../data/interim/sparcs_2022.parquet")
print(df.shape)
print(list(df.columns))

(2103433, 33)
['hospital_service_area', 'hospital_county', 'operating_certificate_number', 'permanent_facility_id', 'facility_name', 'age_group', 'zip_code_-_3_digits', 'gender', 'race', 'ethnicity', 'length_of_stay', 'type_of_admission', 'patient_disposition', 'discharge_year', 'ccsr_diagnosis_code', 'ccsr_diagnosis_description', 'ccsr_procedure_code', 'ccsr_procedure_description', 'apr_drg_code', 'apr_drg_description', 'apr_mdc_code', 'apr_mdc_description', 'apr_severity_of_illness_code', 'apr_severity_of_illness_description', 'apr_risk_of_mortality', 'apr_medical_surgical_description', 'payment_typology_1', 'payment_typology_2', 'payment_typology_3', 'birth_weight', 'emergency_department_indicator', 'total_charges', 'total_costs']


In [2]:
print(df["patient_disposition"].value_counts().head(15))

patient_disposition
Home or Self Care                        1385615
Home w/ Home Health Services              280877
Skilled Nursing Home                      194779
Expired                                    59406
Left Against Medical Advice                59172
Short-term Hospital                        35353
Inpatient Rehabilitation Facility          34691
Hospice - Home                             11700
Psychiatric Hospital or Unit of Hosp        9963
Hospice - Medical Facility                  9441
Facility w/ Custodial/Supportive Care       6241
Another Type Not Listed                     5063
Court/Law Enforcement                       3613
Medicare Cert Long Term Care Hospital       2356
Cancer Center or Children's Hospital        1888
Name: count, dtype: int64


In [4]:
print(df["permanent_facility_id"].nunique(), "hospitals")
print(df["permanent_facility_id"].isna().sum(), "rows with no facility id")

206 hospitals
5390 rows with no facility id


In [5]:
df["y"] = (df["patient_disposition"] == "Expired").astype(int)
g = df.groupby("permanent_facility_id")["y"].agg(n="size", deaths="sum")
g["rate"] = g["deaths"] / g["n"]
print(g.describe())
print((g["deaths"] >= 50).sum(), "hospitals with at least 50 deaths")
print((g["deaths"] >= 100).sum(), "hospitals with at least 100 deaths")

                  n       deaths        rate
count    206.000000   206.000000  206.000000
mean   10184.674757   288.053398    0.035783
std    10761.292201   325.926794    0.084952
min        2.000000     0.000000    0.000000
25%     1976.750000    37.000000    0.018925
50%     7235.500000   192.500000    0.027527
75%    14922.000000   417.000000    0.038830
max    50110.000000  1683.000000    0.893993
145 hospitals with at least 50 deaths
128 hospitals with at least 100 deaths


In [6]:
df = df.dropna(subset=["permanent_facility_id"]).copy()
df["y"] = (df["patient_disposition"] == "Expired").astype(int)
df.to_parquet("../data/interim/sparcs_2022_clean.parquet")

g = df.groupby("permanent_facility_id").agg(
    name=("facility_name", "first"), n=("y", "size"), deaths=("y", "sum"))
g["rate"] = g["deaths"] / g["n"]
g.to_csv("../docs/hospital_summary_2022.csv")
print(g.sort_values("rate", ascending=False).head(8))
print(g.sort_values("n").head(8))

                                                                    name  \
permanent_facility_id                                                      
010223                                                  Calvary Hospital   
001175                                              Calvary Hospital Inc   
001158                 Soldiers and Sailors Memorial Hospital of Yate...   
001486                                Henry J. Carter Specialty Hospital   
000098                                  Brooks-TLC Hospital System, Inc.   
000873                                   Ira Davenport Memorial Hospital   
001293                      New York Community Hospital of Brooklyn, Inc   
000671                                           Geneva General Hospital   

                          n  deaths      rate  
permanent_facility_id                          
010223                  283     253  0.893993  
001175                 1967    1683  0.855618  
001158                  288      22  0.076389  

In [7]:
import numpy as np

df = df[~df["facility_name"].str.contains("Calvary", case=False, na=False)].copy()
df.to_parquet("../data/interim/sparcs_2022_analysis.parquet")

g = df.groupby("permanent_facility_id").agg(n=("y", "size"), deaths=("y", "sum"))
g["rate"] = g["deaths"] / g["n"]
print(len(g), "hospitals after exclusion")

g["n_cert"] = (g["n"] * 0.25).astype(int).clip(lower=1)
g["hw_all"] = 100 * 1.96 * np.sqrt(g["rate"] * (1 - g["rate"]) / g["n_cert"])
g["hw_grp20"] = 100 * 1.96 * np.sqrt(g["rate"] * (1 - g["rate"]) / (g["n_cert"] * 0.2).clip(lower=1))

for cut in [1.0, 1.5, 2.0]:
    print(f"half-width <= {cut} pts: whole hospital {(g.hw_all <= cut).sum()}, a group that is 20% of patients {(g.hw_grp20 <= cut).sum()}")
print(g[["n", "deaths", "hw_all", "hw_grp20"]].describe())

204 hospitals after exclusion
half-width <= 1.0 pts: whole hospital 143, a group that is 20% of patients 59
half-width <= 1.5 pts: whole hospital 168, a group that is 20% of patients 106
half-width <= 2.0 pts: whole hospital 179, a group that is 20% of patients 134
                  n       deaths      hw_all    hw_grp20
count    204.000000   204.000000  204.000000  204.000000
mean   10273.495098   281.387255    1.130034    2.526833
std    10776.028475   312.471685    1.515763    3.389349
min        2.000000     0.000000    0.000000    0.000000
25%     2074.250000    36.750000    0.409399    0.915444
50%     7318.000000   190.500000    0.631805    1.412759
75%    15200.250000   415.000000    1.125167    2.515949
max    50110.000000  1575.000000    8.711111   19.478637


In [8]:
for c in ["gender", "race", "ethnicity", "age_group", "type_of_admission", "emergency_department_indicator"]:
    print("\n", c)
    print(df[c].value_counts(dropna=False).head(12))

print("\ndeaths by age group")
print(df.groupby("age_group")["y"].agg(n="size", deaths="sum", rate="mean"))
print("\ndeaths by race")
print(df.groupby("race")["y"].agg(n="size", deaths="sum", rate="mean"))


 gender
gender
F    1145588
M     950079
U        126
Name: count, dtype: int64

 race
race
White                     1121449
Other Race                 565290
Black/African American     382227
Multi-racial                26827
Name: count, dtype: int64

 ethnicity
ethnicity
Not Span/Hispanic    1528283
Spanish/Hispanic      324620
Unknown               238097
Multi-ethnic            4793
Name: count, dtype: int64

 age_group
age_group
70 or Older    642303
50 to 69       565358
30 to 49       402239
0 to 17        301633
18 to 29       184260
Name: count, dtype: int64

 type_of_admission
type_of_admission
Emergency        1390474
Elective          352947
Newborn           198615
Urgent            144695
Trauma              7301
Not Available       1761
Name: count, dtype: int64

 emergency_department_indicator
emergency_department_indicator
Y    1333656
N     762137
Name: count, dtype: int64

deaths by age group
                  n  deaths      rate
age_group                         

In [9]:
df = df[df["age_group"] != "0 to 17"].copy()
df = df[df["gender"].isin(["F", "M"])].copy()
df["age_band"] = df["age_group"].replace({"18 to 29": "18-49", "30 to 49": "18-49"})
df["race_grp"] = df["race"].replace({"Multi-racial": "Other/Multi", "Other Race": "Other/Multi"})
df.to_parquet("../data/interim/sparcs_2022_adults.parquet")
print(len(df), "adult rows;", df["y"].sum(), "deaths;", df["permanent_facility_id"].nunique(), "hospitals")

parts = []
for var in ["gender", "age_band", "race_grp"]:
    t = df.groupby(["permanent_facility_id", var])["y"].agg(n="size", deaths="sum").reset_index()
    t = t.rename(columns={var: "group"})
    t["var"] = var
    parts.append(t)
gr = pd.concat(parts)
gr["cert_deaths"] = gr["deaths"] * 0.25

for var in ["gender", "age_band", "race_grp"]:
    t = gr[gr["var"] == var]
    for thr in [10, 20, 40]:
        print(f"{var:9s} thr {thr:2d}: {(t['cert_deaths'] >= thr).sum()} of {len(t)} hospital-group cells testable")

print(df.groupby("permanent_facility_id")["race_grp"].apply(lambda s: (s == "Other/Multi").mean()).describe())

1794116 adult rows; 56609 deaths; 204 hospitals
gender    thr 10: 266 of 404 hospital-group cells testable
gender    thr 20: 218 of 404 hospital-group cells testable
gender    thr 40: 139 of 404 hospital-group cells testable
age_band  thr 10: 277 of 600 hospital-group cells testable
age_band  thr 20: 202 of 600 hospital-group cells testable
age_band  thr 40: 121 of 600 hospital-group cells testable
race_grp  thr 10: 272 of 585 hospital-group cells testable
race_grp  thr 20: 206 of 585 hospital-group cells testable
race_grp  thr 40: 120 of 585 hospital-group cells testable
count    204.000000
mean       0.182811
std        0.170989
min        0.000000
25%        0.052383
50%        0.124783
75%        0.279968
max        0.747667
Name: race_grp, dtype: float64


In [10]:
import numpy as np, pandas as pd

df = pd.read_parquet("../data/interim/sparcs_2022_adults.parquet").reset_index(drop=True)
hosp = df.groupby("permanent_facility_id")["y"].agg(n="size", deaths="sum")
for cut in [50, 100, 200]:
    print(f"at least {cut} deaths: {(hosp['deaths'] >= cut).sum()} hospitals")

for seed in range(5):
    rng = np.random.default_rng(seed)
    u = pd.Series(rng.random(len(df)))
    r = u.groupby([df["permanent_facility_id"], df["y"]]).rank(pct=True)
    split = np.where(r <= 0.50, "train", np.where(r <= 0.75, "cert", "test"))
    out = pd.DataFrame({"row": df.index, "hospital": df["permanent_facility_id"], "split": split})
    out.to_parquet(f"../data/splits/split_seed{seed}.parquet")

print(out["split"].value_counts(normalize=True))
print(out.assign(y=df["y"]).groupby("split")["y"].mean())

at least 50 deaths: 142 hospitals
at least 100 deaths: 126 hospitals
at least 200 deaths: 99 hospitals
split
train    0.499944
test     0.250083
cert     0.249973
Name: proportion, dtype: float64
split
cert     0.031502
test     0.031698
train    0.031505
Name: y, dtype: float64
